# 03c CFPB Time Series Deep Learning (NeuralProphet)

Objective: Build NeuralProphet baselines for quarterly Credit Reporting complaints using the same y and 33/10/10 split framing from 03a/03b, with 4-quarter-ahead evaluation.

In [2]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neuralprophet import NeuralProphet

warnings.filterwarnings("ignore")

Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [3]:
# ============================================================================
# DATA SETUP (MATCH 03a/03b)
# ============================================================================
cols = ["Product", "year_quarter", "Date received"]
df = pd.read_parquet("../data/processed/full_dataset.parquet", columns=cols)

if "year_quarter" not in df.columns:
    df["Date received"] = pd.to_datetime(df["Date received"], errors="coerce")
    df["year_quarter"] = df["Date received"].dt.to_period("Q").astype(str)

mask_cr = df["Product"].str.contains("Credit reporting", case=False, na=False)
series_df = (
    df.loc[mask_cr]
      .groupby("year_quarter")
      .size()
      .rename("y")
      .to_frame()
      .sort_index()
)
series_df.index = pd.PeriodIndex(series_df.index, freq="Q")
series_df = series_df.loc[:"2025Q4"].copy()
y = series_df["y"].astype(float)

# Same split logic as 03a/03b target split (expected 33/10/10 for this y window).
n = len(y)
test_size = max(4, n // 5)
val_size = test_size
while n - (val_size + test_size) < 8 and test_size > 2:
    test_size -= 1
    val_size = test_size

train = y.iloc[: n - (val_size + test_size)]
val = y.iloc[n - (val_size + test_size) : n - test_size]
test = y.iloc[n - test_size :]

print(f"Points: {len(y)} | train={len(train)}, val={len(val)}, test={len(test)}")
print(f"Window: {y.index.min()} -> {y.index.max()}")

Points: 53 | train=33, val=10, test=10
Window: 2012Q4 -> 2025Q4


In [ ]:
# ============================================================================
# SHARED EVAL HELPERS
# ============================================================================
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    ratio = np.where(denom == 0, 0.0, 2.0 * np.abs(y_true - y_pred) / denom)
    return float(100.0 * np.mean(ratio))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

y_val_multi = None
y_test_multi = None

def append_result(model_name, val_pred, test_pred, runtime_sec):
    if np.ndim(val_pred) == 2:
        if y_val_multi is None or y_test_multi is None:
            raise ValueError("y_val_multi and y_test_multi must be built before multi-horizon evaluation.")
        v_scores = [
            (
                smape(y_val_multi.iloc[:, h].values, val_pred[:, h]),
                mae(y_val_multi.iloc[:, h].values, val_pred[:, h]),
                rmse(y_val_multi.iloc[:, h].values, val_pred[:, h])
            )
            for h in range(val_pred.shape[1])
        ]
        t_scores = [
            (
                smape(y_test_multi.iloc[:, h].values, test_pred[:, h]),
                mae(y_test_multi.iloc[:, h].values, test_pred[:, h]),
                rmse(y_test_multi.iloc[:, h].values, test_pred[:, h])
            )
            for h in range(test_pred.shape[1])
        ]
        smape_val, mae_val, rmse_val = np.mean(v_scores, axis=0)
        smape_test, mae_test, rmse_test = np.mean(t_scores, axis=0)
    else:
        smape_val = smape(val.values, val_pred)
        mae_val = mae(val.values, val_pred)
        rmse_val = rmse(val.values, val_pred)
        smape_test = smape(test.values, test_pred)
        mae_test = mae(test.values, test_pred)
        rmse_test = rmse(test.values, test_pred)

    eval_rows.append({
        "Model": model_name,
        "sMAPE_val": smape_val,
        "MAE_val": mae_val,
        "RMSE_val": rmse_val,
        "sMAPE_test": smape_test,
        "MAE_test": mae_test,
        "RMSE_test": rmse_test,
        "runtime_sec": runtime_sec
    })

def plot_section_predictions(section_title, pred_map):
    fig, ax = plt.subplots(figsize=(10, 4))
    y_train_ts = pd.Series(train.values, index=train.index.to_timestamp(how="end"))
    val_idx = val.index.to_timestamp(how="end")
    test_idx = test.index.to_timestamp(how="end")

    ax.plot(y_train_ts.index, y_train_ts.values, color="steelblue", label="Observed - Train")
    ax.plot(val_idx, val.values, color="steelblue", linestyle="--", alpha=0.8, label="Observed - Val (h=1)")
    ax.plot(test_idx, test.values, color="steelblue", linestyle=":", alpha=0.9, label="Observed - Test (h=1)")

    for model_name, preds in pred_map.items():
        val_pred_h1, test_pred_h1 = preds
        ax.plot(val_idx, val_pred_h1, linestyle="--", marker="o", label=f"{model_name} - Val (h=1)")
        ax.plot(test_idx, test_pred_h1, marker="o", label=f"{model_name} - Test (h=1)")

    ax.set_title(section_title)
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

def to_np_df(s):
    return pd.DataFrame({"ds": s.index.to_timestamp(how="start"), "y": s.values})

def recursive_h1_with_actuals(model, history_df, actual_series):
    hist = history_df.copy()
    preds = []
    for ts, y_true in zip(actual_series.index.to_timestamp(how="start"), actual_series.values):
        fut = model.make_future_dataframe(hist, periods=1, n_historic_predictions=False)
        fc = model.predict(fut)
        yhat = float(fc["yhat1"].dropna().iloc[-1])
        preds.append(yhat)
        hist = pd.concat([hist, pd.DataFrame({"ds": [ts], "y": [y_true]})], ignore_index=True)
    return np.asarray(preds, dtype=float)

def recursive_h_forecast_single(model, origin_history_df, h=4):
    hist = origin_history_df.copy()
    out = []
    for _ in range(h):
        fut = model.make_future_dataframe(hist, periods=1, n_historic_predictions=False)
        fc = model.predict(fut)
        yhat = float(fc["yhat1"].dropna().iloc[-1])
        out.append(yhat)
        next_ds = (pd.Period(hist["ds"].iloc[-1], freq="Q") + 1).to_timestamp(how="start")
        hist = pd.concat([hist, pd.DataFrame({"ds": [next_ds], "y": [yhat]})], ignore_index=True)
    return np.asarray(out, dtype=float)

def direct_h_forecast_multi(model, origin_history_df, h=4):
    fut = model.make_future_dataframe(origin_history_df, periods=h, n_historic_predictions=False)
    fc = model.predict(fut)
    cols = [f"yhat{i}" for i in range(1, h + 1)]
    row = None
    for i in range(len(fc) - 1, -1, -1):
        cand = fc.iloc[i][cols]
        if cand.notna().all():
            row = cand
            break
    if row is None:
        row = fc[cols].ffill().iloc[-1]
    return row.values.astype(float)

def build_truth_matrix(split_series, h=4, n_origins=4):
    rows = []
    for i in range(n_origins):
        rows.append([float(split_series.iloc[i + j]) for j in range(h)])
    df_out = pd.DataFrame(rows, columns=[f"y_h{k}" for k in range(1, h + 1)])
    return df_out

eval_rows = []
np_h1_rows = []

In [5]:
# ============================================================================
# BUILD NP DATAFRAMES AND H=4 TRUTH TABLES
# ============================================================================
df_train = to_np_df(train)
df_val = to_np_df(val)
df_test = to_np_df(test)
df_train_val = to_np_df(pd.concat([train, val]))
df_all = to_np_df(y)

# For 4-quarter direct evaluation, use first 4 rolling origins in val and test windows.
y_val_multi = build_truth_matrix(val, h=4, n_origins=4)
y_test_multi = build_truth_matrix(test, h=4, n_origins=4)

display(y_val_multi.head())
display(y_test_multi.head())

,y_h1,y_h2,y_h3,y_h4
0,68462.0,77872.0,80799.0,78278.0
1,77872.0,80799.0,78278.0,116089.0
2,80799.0,78278.0,116089.0,147089.0
3,78278.0,116089.0,147089.0,150013.0


,y_h1,y_h2,y_h3,y_h4
0,276130.0,287864.0,398641.0,516605.0
1,287864.0,398641.0,516605.0,661366.0
2,398641.0,516605.0,661366.0,775293.0
3,516605.0,661366.0,775293.0,1007034.0


## 1. Single-Step Baseline (AR + trend/seasonal)

In [ ]:
# ============================================================================
# SECTION 1: NP SINGLE-STEP
# ============================================================================
t0 = time.perf_counter()
m = NeuralProphet(
    n_forecasts=1,
    n_lags=6,
    weekly_seasonality=False,
    yearly_seasonality=True,
    learning_rate=0.1,
    ar_reg=0.5,
    trend_reg=0.1,
    trend_reg_threshold=True
)
_ = m.fit(df_train, freq="Q", validation_df=df_val, progress="off")
runtime_single = time.perf_counter() - t0

# h=1 walk-forward predictions over full val and test windows (length 10 each).
np_single_val_h1 = recursive_h1_with_actuals(m, df_train, val)
np_single_test_h1 = recursive_h1_with_actuals(m, df_train_val, test)

# h=1 metrics for comparison section.
np_h1_rows.append({
    "Model": "NP-Single (h=1)",
    "sMAPE_val": smape(val.values, np_single_val_h1),
    "MAE_val": mae(val.values, np_single_val_h1),
    "RMSE_val": rmse(val.values, np_single_val_h1),
    "sMAPE_test": smape(test.values, np_single_test_h1),
    "MAE_test": mae(test.values, np_single_test_h1),
    "RMSE_test": rmse(test.values, np_single_test_h1),
    "runtime_sec": runtime_single
})

# h=4 recursive-from-single predictions for first 4 origins in val and test.
single_val_rows = []
for i in range(4):
    hist = to_np_df(pd.concat([train, val.iloc[:i]]))
    single_val_rows.append(recursive_h_forecast_single(m, hist, h=4))
np_single_val_h4 = np.vstack(single_val_rows)

single_test_rows = []
for i in range(4):
    hist = to_np_df(pd.concat([train, val, test.iloc[:i]]))
    single_test_rows.append(recursive_h_forecast_single(m, hist, h=4))
np_single_test_h4 = np.vstack(single_test_rows)

append_result("NP-Single", np_single_val_h4, np_single_test_h4, runtime_single)
print("NP-Single complete.")

INFO - (NP.config.__post_init__) - Note: Trend changepoint regularization is experimental.
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency QS corresponds to 78.788% of the data.


INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - QS
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 8
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 430


Training: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency QS corresponds to 79.592% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - QS
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency QS corresponds to 85.714% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - QS
INFO - (NP.df_utils._infer_frequency) - Major frequency QS corresponds to 85.714% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - QS
INFO - (NP.data.processing._handle_missing_data) - Dropped 1 rows at the end with NaNs in 'y' column.


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


NP-Single complete.


## 2. Multi-Horizon Direct (n_forecasts=4)

In [11]:
# ============================================================================
# SECTION 2: NP MULTI-HORIZON DIRECT
# ============================================================================
t0 = time.perf_counter()
m_multi = NeuralProphet(
    n_forecasts=4,
    n_lags=6,
    weekly_seasonality=False,
    yearly_seasonality=True,
    learning_rate=0.1,
    ar_reg=0.5,
    trend_reg=0.1,
    trend_reg_threshold=True
)
_ = m_multi.fit(df_train, freq="Q", validation_df=df_val, progress="off")
runtime_multi = time.perf_counter() - t0

# h=1 walk-forward (for section plot + comparison) using direct model's yhat1.
def h1_from_direct(model, history_df, actual_series):
    hist = history_df.copy()
    preds = []
    for ts, y_true in zip(actual_series.index.to_timestamp(how="start"), actual_series.values):
        vec = direct_h_forecast_multi(model, hist, h=4)
        preds.append(float(vec[0]))
        hist = pd.concat([hist, pd.DataFrame({"ds": [ts], "y": [y_true]})], ignore_index=True)
    return np.asarray(preds, dtype=float)

np_multi_val_h1 = h1_from_direct(m_multi, df_train, val)
np_multi_test_h1 = h1_from_direct(m_multi, df_train_val, test)

np_h1_rows.append({
    "Model": "NP-Multi (h=1)",
    "sMAPE_val": smape(val.values, np_multi_val_h1),
    "MAE_val": mae(val.values, np_multi_val_h1),
    "RMSE_val": rmse(val.values, np_multi_val_h1),
    "sMAPE_test": smape(test.values, np_multi_test_h1),
    "MAE_test": mae(test.values, np_multi_test_h1),
    "RMSE_test": rmse(test.values, np_multi_test_h1),
    "runtime_sec": runtime_multi
})

# h=4 direct predictions for first 4 origins in val and test.
multi_val_rows = []
for i in range(4):
    hist = to_np_df(pd.concat([train, val.iloc[:i]]))
    multi_val_rows.append(direct_h_forecast_multi(m_multi, hist, h=4))
np_multi_val_h4 = np.vstack(multi_val_rows)

multi_test_rows = []
for i in range(4):
    hist = to_np_df(pd.concat([train, val, test.iloc[:i]]))
    multi_test_rows.append(direct_h_forecast_multi(m_multi, hist, h=4))
np_multi_test_h4 = np.vstack(multi_test_rows)

append_result("NP-Multi", np_multi_val_h4, np_multi_test_h4, runtime_multi)
plot_section_predictions("NP Single vs Multi (h=1 shown)", {"NP-Single": (np_single_val_h1, np_single_test_h1), "NP-Multi": (np_multi_val_h1, np_multi_test_h1)})

results = pd.DataFrame(eval_rows).copy()
for c in ["sMAPE_val", "MAE_val", "RMSE_val", "sMAPE_test", "MAE_test", "RMSE_test", "runtime_sec"]:
    results[c] = results[c].round(3)
display(results)

INFO - (NP.config.__post_init__) - Note: Trend changepoint regularization is experimental.
WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency QS corresponds to 78.788% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - QS
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 8
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 460


Training: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

: 

## 3. Time-Series Cross-Validation

In [ ]:
# ============================================================================
# SECTION 3: CV (NeuralProphet crossvalidation API varies by version)
# ============================================================================
cv_metrics_df = None
try:
    from neuralprophet import crossvalidation
    cv_out = crossvalidation(m_multi, df_all, freq="Q", fold_pct=0.1, fold_overlap_pct=0.0)
    if isinstance(cv_out, (list, tuple)) and len(cv_out) > 1:
        cv_metrics_df = cv_out[1]
    elif isinstance(cv_out, pd.DataFrame):
        cv_metrics_df = cv_out
except Exception as e:
    print("crossvalidation() not available with current signature/version:", e)

if cv_metrics_df is not None:
    print("CV metrics (raw):")
    display(cv_metrics_df.head(20))
else:
    print("CV metrics unavailable via built-in API in this environment; proceed with holdout metrics above.")

## 4. Interpretability (Components + AR/Parameters)

In [ ]:
# ============================================================================
# SECTION 4: INTERPRETABILITY
# ============================================================================
forecast_for_components = m_multi.predict(m_multi.make_future_dataframe(df_train, periods=4, n_historic_predictions=True))
_ = m_multi.plot_components(forecast_for_components)
try:
    _ = m_multi.plot_parameters()
except Exception as e:
    print("plot_parameters not available in this version:", e)

# Attempt AR weight extraction (version-dependent internals).
ar_weight_df = None
try:
    if hasattr(m_multi.model, "ar_weights"):
        w = m_multi.model.ar_weights.detach().cpu().numpy().reshape(-1)
        ar_weight_df = pd.DataFrame({"lag": np.arange(1, len(w) + 1), "weight": w})
        ar_weight_df["abs_weight"] = np.abs(ar_weight_df["weight"])
        ar_weight_df = ar_weight_df.sort_values("abs_weight", ascending=False).reset_index(drop=True)
except Exception as e:
    print("AR weight extraction unavailable in current NP internals:", e)

if ar_weight_df is not None:
    print("Top AR lags by |weight|:")
    display(ar_weight_df.head(12))
else:
    print("AR weight table not extracted; use component/parameter plots above.")

## 5. Compare vs 03b XGB/RF Baselines (h=1)

In [ ]:
# ============================================================================
# SECTION 5: COMPARISON VS 03b TREE H1
# ============================================================================
np_h1_df = pd.DataFrame(np_h1_rows).copy()
np_h1_df["Family"] = "neuralprophet-h1"

tree_h1_path = "../reports/2026-03-25/tables/03b_tree_h1_summary.csv"
if os.path.exists(tree_h1_path):
    tree_h1_df = pd.read_csv(tree_h1_path)
    combined_h1 = pd.concat([tree_h1_df, np_h1_df], ignore_index=True, sort=False)
else:
    print(f"Warning: {tree_h1_path} not found. Showing NeuralProphet h=1 only.")
    combined_h1 = np_h1_df.copy()

for c in ["sMAPE_val", "MAE_val", "RMSE_val", "sMAPE_test", "MAE_test", "RMSE_test", "runtime_sec"]:
    combined_h1[c] = pd.to_numeric(combined_h1[c], errors="coerce").round(3)

combined_h1 = combined_h1.sort_values(["sMAPE_test", "MAE_test", "runtime_sec"]).reset_index(drop=True)
display(combined_h1)

## 6. Save Multi-Horizon Model Artifact for 04 South

In [ ]:
# ============================================================================
# SECTION 6: SAVE MODEL
# ============================================================================
import torch

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

state_path = f"{model_dir}/03c_np_multi_state_dict.pt"
torch.save(m_multi.model.state_dict(), state_path)

# Save config and key metrics for reproducible reload setup.
meta_path = f"{model_dir}/03c_np_multi_meta.json"
pd.Series({
    "n_forecasts": 4,
    "n_lags": 6,
    "freq": "Q",
    "train_points": len(train),
    "val_points": len(val),
    "test_points": len(test)
}).to_json(meta_path, indent=2)

print("Saved NeuralProphet model state:", state_path)
print("Saved metadata:", meta_path)